In [1]:
## Vloco de codigo para instalar versao especifica dos pacotes
##pip install pandas==1.5.3
##pip install numpy==1.24.3

In [2]:
import sys
import os

# Acesso aos módulos do diretório
from pathlib import Path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))
print("Project root:", project_root)

Project root: C:\pod\hackathon_pod_2025


##### Carregando pacotes

In [3]:
# Pacotes de manipulacao

import pandas as pd
import numpy as np
import os

# Pacotes de visualizacao
import matplotlib.pyplot as plt
import seaborn as sns

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

pandas: 1.5.3
numpy: 1.26.4


## Carregando databases

#### Base Dados Cadastrais

In [4]:
## Carregando todos arquivos em parquet de uma pasta

#path = project_root +'database/raw/base_score_bureau_movel/base_score_bureau_movel/'

all_files = [os.path.join(project_root/'database/raw/base_score_bureau_movel/', f) for f in os.listdir(project_root/'database/raw/base_score_bureau_movel/') if f.endswith('.parquet')]
df_list = [pd.read_parquet(f, engine='pyarrow') for f in all_files]

df_bureau = pd.concat(df_list, ignore_index=True)

In [5]:
df_bureau.head()

,SAFRA,FLAG_INSTALACAO,FPD,PROD,flag_mig2,SCORE_01,SCORE_02,NUM_CPF
0,202410,1,0,CMV,PRE,562,636,ZZZZZX7XWY8
1,202410,1,1,CMV,PRE,546,518,ZZZZZX88YXY
2,202410,1,0,CMV,PRE,621,750,ZZZZZYT7XYT
3,202410,1,1,CMV,PRE,609,679,ZZZZZNTXY9Z
4,202410,1,0,CMV,PRE,621,722,ZZZZZ79ZXUX


In [6]:
df_bureau.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1290526 entries, 0 to 1290525
Data columns (total 8 columns):
 #   Column           Non-Null Count    Dtype 
---  ------           --------------    ----- 
 0   SAFRA            1290526 non-null  object
 1   FLAG_INSTALACAO  1290526 non-null  object
 2   FPD              1290526 non-null  object
 3   PROD             1290526 non-null  object
 4   flag_mig2        1290526 non-null  object
 5   SCORE_01         1281087 non-null  object
 6   SCORE_02         1289950 non-null  object
 7   NUM_CPF          1290526 non-null  object
dtypes: object(8)
memory usage: 78.8+ MB


#### Ajustando os tipos de dados

In [7]:
# Iremos ajustar os tipos de dados para otimizar a memoria e o correto processamento
df_bureau['SAFRA'] = df_bureau['SAFRA'].astype('object')
df_bureau['FLAG_INSTALACAO'] = df_bureau['FLAG_INSTALACAO'].astype('int')
df_bureau['FPD'] = df_bureau['FPD'].astype('int')
df_bureau['PROD'] = df_bureau['PROD'].astype('object')
df_bureau['flag_mig2'] = df_bureau['flag_mig2'].astype('object')
df_bureau['SCORE_01'] = df_bureau['SCORE_01'].astype('float64')
df_bureau['SCORE_02'] = df_bureau['SCORE_02'].astype('float64')
df_bureau['NUM_CPF'] = df_bureau['NUM_CPF'].astype('object')

#### Feature Engineer

Iremos criar apenas as variáveis:
- Relação entre `SCORE_01` e `SCORE_02`
- Média entre `SCORE_01` e `SCORE_02`
- Diferenca entre `SCORE_01` e `SCORE_02`
- Minino entre `SCORE_01` e `SCORE_02`

In [8]:
# Iremos criar as medidas descritas acima
df_bureau['SCORE_RATEO'] = df_bureau['SCORE_02'] / df_bureau['SCORE_01']
df_bureau['SCORE_AVG'] = (df_bureau['SCORE_01'] + df_bureau['SCORE_02']) / 2
df_bureau['SCORE_DIFF'] = df_bureau['SCORE_02'] - df_bureau['SCORE_01']
df_bureau['SCORE_MIN'] = df_bureau[['SCORE_01', 'SCORE_02']].min(axis=1)

In [9]:
# Criando novo dataset
book_variaveis_01 = df_bureau.copy()

In [10]:
# Salvando o dataframe em parquet
book_variaveis_01.to_parquet(project_root/'database/processed/book_variaveis_01.parquet', index=False)